# Insurance Claims Fine-Tuning — Full Nebius Token Factory Pipeline

**Why this matters:** Insurance customer service needs accurate terminology (coverage, deductibles, intake steps) and consistent answers. A general-purpose model can sound fluent yet be wrong on domain details. Fine-tuning on insurance-oriented dialogue teaches the model your style and vocabulary; a stronger **teacher** model can further improve the training targets before you train the smaller **student** with LoRA.

**What this notebook does (scope):** This is an **end-to-end learning demo** on the [Bitext Insurance LLM Chatbot](https://huggingface.co/datasets/bitext/Bitext-insurance-llm-chatbot-training-dataset) dataset: Data Lab upload, **70B teacher** batch inference, curated JSONL, **LoRA** fine-tune on **8B**, **serverless** adapter deployment, and a **before/after** comparison. It does **not** implement a full production stack (separate triage, extraction, and adjudication models)—that is a natural next step once you own the data and compliance story.

This notebook runs an **end-to-end** workflow:

1. Load the Bitext dataset (default **~32** rows — quick demo ~5–7 min; configurable).
2. **Before** — run the base **8B** model on 10 held-out questions.
3. **Data Lab** — upload rows and run **batch inference** with a **70B teacher**.
4. **Curate** teacher outputs into conversational JSONL for fine-tuning.
5. **LoRA fine-tune** `meta-llama/Llama-3.1-8B-Instruct` (training API id).
6. **Deploy** the adapter as a **serverless custom model** (`POST /v0/models`).
7. **After** — run the same 10 questions on the **fine-tuned** endpoint.

> Tip: run cells in order. State is saved under `insurance_claims_demo_artifacts/`. See [README.md](README.md) for setup, diagrams, and references.


## Before you run

### What this notebook teaches

You will wire together **Data Lab** (datasets + batch jobs), **fine-tuning** (OpenAI-compatible jobs + LoRA), **custom model deployment** (serverless adapter), and a small **evaluation** loop—then optionally run the **Gradio** app in `app.py` to compare base vs fine-tuned chat.

### What you need

- **Nebius API key** with access to Data Lab, batch inference, fine-tuning, and custom models ([Token Factory](https://tokenfactory.nebius.com/) → project settings).
- **Python 3.10+** and a Jupyter environment (local Jupyter, VS Code, or Colab).
- **Patience** — batch inference and fine-tuning can take a while depending on queue load.

Install dependencies in the next cell, then enter your API key.

> Run **top to bottom** the first time. After that, you can resume from saved state in `insurance_claims_demo_artifacts/` (see step cells).


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai", "requests", "datasets"])

In [ ]:
import getpass
import os

if not os.environ.get("NEBIUS_API_KEY"):
    os.environ["NEBIUS_API_KEY"] = getpass.getpass("Enter your NEBIUS_API_KEY: ")

print("API key loaded:", bool(os.environ.get("NEBIUS_API_KEY")))


## Step 0 — Setup and configuration

**What we are doing:** This cell is the **control panel**: API base URLs, teacher and base model names, LoRA hyperparameters, artifact paths, and demo size (`SAMPLE_SIZE`, `N_EPOCHS`).

**Why we do it:** One place to tune the run (speed vs quality) and to keep IDs consistent—Nebius uses different model strings for **fine-tuning** vs **chat inference** (see prints below).

**In simple words:** Turn the knobs here before running the pipeline.


In [ ]:
import json
import os
import random
import time
import uuid
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any
from urllib.parse import quote

import requests
from datasets import load_dataset
from openai import OpenAI

API_KEY = os.environ["NEBIUS_API_KEY"]
BASE_URL = "https://api.tokenfactory.nebius.com/v1/"
CONTROL_URL = "https://api.tokenfactory.nebius.com"

TEACHER_MODEL = "meta-llama/Llama-3.3-70B-Instruct"
# Fine-tuning API enum uses this id; chat completions use CHAT_BASE_MODEL (inference registry).
FINE_TUNE_BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
CHAT_BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
# Deploy may accept either the fine-tuning id or the Meta-prefixed chat id; try both.
DEPLOY_BASE_MODEL_CANDIDATES = [
    FINE_TUNE_BASE_MODEL,
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
]

LORA_RANK = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
N_EPOCHS = 1
POLL_SECONDS = 20
BATCH_COMPLETION_WINDOW = "24h"
DATA_LAB_FOLDER = "/demo/insurance-claims"
ADAPTER_NAME = "astro-insurance"

ARTIFACT_DIR = Path("insurance_claims_demo_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
STATE_PATH = ARTIFACT_DIR / "insurance_notebook_state.json"
RAW_DATASET_PATH = ARTIFACT_DIR / "insurance_raw_dataset.jsonl"
RAW_BATCH_OUTPUT_PATH = ARTIFACT_DIR / "insurance_batch_output.jsonl"
CURATED_TRAINING_PATH = ARTIFACT_DIR / "insurance_curated_training.jsonl"
TEST_PROMPTS_PATH = ARTIFACT_DIR / "test_prompts.json"
BEFORE_ANSWERS_PATH = ARTIFACT_DIR / "before_answers.json"
AFTER_ANSWERS_PATH = ARTIFACT_DIR / "after_answers.json"
TRAINING_SAMPLE_META_PATH = ARTIFACT_DIR / "training_sample_meta.json"
TRAIN_SAMPLE_PATH = ARTIFACT_DIR / "train_sample.json"

# Optional: skip Step 4 Data Lab batch — reuse a previous successful teacher output dataset.
REUSE_OUTPUT_DATASET_ID = os.environ.get("REUSE_OUTPUT_DATASET_ID", "").strip()
REUSE_BATCH_FROM_STATE = os.environ.get("REUSE_BATCH_FROM_STATE", "").strip().lower() in (
    "1",
    "true",
    "yes",
)

# Demo: ~5-7 min target for batch + fine-tune (light queue). Raise SAMPLE_SIZE / N_EPOCHS for quality.
SAMPLE_SIZE = 32
RANDOM_SEED = 42

CLAIMS_SYSTEM_PROMPT = (
    "You are a helpful insurance assistant. "
    "You explain coverage, claims intake, deductibles, and next steps clearly and accurately. "
    "You do not provide legal advice or guarantee claim outcomes. "
    "When policy-specific details are needed, direct the customer to their policy documents or a licensed adjuster. "
    "Be concise, empathetic, and consistent with standard P&C insurance terminology."
)

TEACHER_SYSTEM_PROMPT = (
    "You are a senior insurance claims and policy expert. "
    "Answer clearly and accurately for insurance customer service. "
    "Replace template placeholders with natural language (do not output {{PLACEHOLDER}} tokens). "
    "Ask only for the minimum information needed to proceed."
)

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
HEADERS = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

print("Artifacts:", ARTIFACT_DIR.resolve())
print("Adapter name:", ADAPTER_NAME)
print("Teacher:", TEACHER_MODEL)
print("Fine-tune base:", FINE_TUNE_BASE_MODEL)
print("Chat / BEFORE (base inference):", CHAT_BASE_MODEL)
print("Training rows (SAMPLE_SIZE):", SAMPLE_SIZE)
print("Fine-tune epochs (N_EPOCHS):", N_EPOCHS)
if REUSE_OUTPUT_DATASET_ID:
    print("Step 4: will skip new batch (REUSE_OUTPUT_DATASET_ID is set).")
elif REUSE_BATCH_FROM_STATE:
    print("Step 4: will reuse output_dataset_id from state if present (REUSE_BATCH_FROM_STATE).")

## Helper functions

**What we are doing:** One code cell defines reusable functions for **Data Lab** REST calls (`/v1/datasets`, `/v1/operations`), **export + curation** to training JSONL, **OpenAI-compatible** fine-tuning (`client.fine_tuning.jobs`), and **LoRA deployment** (`POST /v0/models`).

**Why we do it:** Keeps later steps short and makes reruns easier when resuming from `insurance_notebook_state.json`.

Run this cell **once** after Step 0.


In [ ]:
def load_state() -> dict[str, Any]:
    if not STATE_PATH.exists():
        return {}
    return json.loads(STATE_PATH.read_text())

def save_state(state: dict[str, Any]) -> None:
    STATE_PATH.write_text(json.dumps(state, indent=2, sort_keys=True) + "\n")

def datalab_request(
    method: str,
    path: str,
    *,
    json_body: dict[str, Any] | None = None,
    params: dict[str, Any] | None = None,
) -> requests.Response:
    response = requests.request(
        method,
        f"{CONTROL_URL}{path}",
        headers=HEADERS,
        json=json_body,
        params=params,
        timeout=120,
    )
    response.raise_for_status()
    return response

def teacher_messages(user_text: str) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": TEACHER_SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]

def extract_text_content(value: Any) -> str:
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):
        parts = [extract_text_content(item) for item in value]
        return "\n".join([part for part in parts if part]).strip()
    if isinstance(value, dict):
        if isinstance(value.get("text"), str):
            return value["text"].strip()
        for key in ("content", "message"):
            text = extract_text_content(value.get(key))
            if text:
                return text
    return ""

def extract_prompt_from_record(record: dict[str, Any], id_map: dict[str, str]) -> str:
    custom_id = str(record.get("custom_id") or "").strip()
    if custom_id and custom_id in id_map:
        return id_map[custom_id]
    for key in ("messages", "completed_dialogue", "prompt"):
        value = record.get(key)
        if isinstance(value, list):
            for message in value:
                if isinstance(message, dict) and message.get("role") == "user":
                    text = extract_text_content(message.get("content"))
                    if text:
                        return text
        elif isinstance(value, str) and value.strip():
            return value.strip()
    return ""

def extract_reply_from_record(record: dict[str, Any]) -> str:
    for key in ("completion", "assistant_response", "output_text", "text"):
        text = extract_text_content(record.get(key))
        if text:
            return text
    response = record.get("response")
    if isinstance(response, dict):
        body = response.get("body") if isinstance(response.get("body"), dict) else response
        choices = body.get("choices") if isinstance(body, dict) else None
        if isinstance(choices, list) and choices:
            choice = choices[0]
            if isinstance(choice, dict):
                message = choice.get("message") if isinstance(choice.get("message"), dict) else None
                if message:
                    text = extract_text_content(message.get("content"))
                    if text:
                        return text
                text = extract_text_content(choice.get("text"))
                if text:
                    return text
        output = body.get("output") if isinstance(body, dict) else None
        if isinstance(output, list):
            for item in output:
                if isinstance(item, dict):
                    text = extract_text_content(item.get("content"))
                    if text:
                        return text
    for key in ("completed_dialogue", "messages"):
        value = record.get(key)
        if isinstance(value, list):
            assistant_messages = [
                extract_text_content(message.get("content"))
                for message in value
                if isinstance(message, dict) and message.get("role") == "assistant"
            ]
            assistant_messages = [msg for msg in assistant_messages if msg]
            if assistant_messages:
                return assistant_messages[-1]
    return ""

def load_id_map_from_artifact() -> dict[str, str]:
    if not RAW_DATASET_PATH.exists():
        raise RuntimeError(f"Missing {RAW_DATASET_PATH}. Run Step 3 first.")
    id_map: dict[str, str] = {}
    with RAW_DATASET_PATH.open() as handle:
        for line in handle:
            if not line.strip():
                continue
            record = json.loads(line)
            cid = record.get("custom_id")
            prompt = record.get("prompt")
            if isinstance(cid, str) and isinstance(prompt, str):
                id_map[cid] = prompt
    return id_map

def deployment_base_models(ft_job_id: str) -> list[str]:
    candidates: list[str] = []
    try:
        job = client.fine_tuning.jobs.retrieve(ft_job_id)
        job_model = getattr(job, "model", None)
        if isinstance(job_model, str) and job_model and job_model not in candidates:
            candidates.append(job_model)
    except Exception as exc:
        print(f"Warning: could not retrieve fine-tuning job model: {exc}")
    for model_name in DEPLOY_BASE_MODEL_CANDIDATES:
        if model_name not in candidates:
            candidates.append(model_name)
    return candidates

def post_dataset_payload(name: str, schema: list[dict], rows: list[dict]) -> dict:
    body = {"name": name, "folder": DATA_LAB_FOLDER, "rows": rows}
    body["schema"] = schema
    try:
        return datalab_request("POST", "/v1/datasets", json_body=body).json()
    except requests.HTTPError as exc:
        if exc.response is not None and exc.response.status_code >= 400:
            alt = {k: v for k, v in body.items() if k != "schema"}
            alt["dataset_schema"] = schema
            return datalab_request("POST", "/v1/datasets", json_body=alt).json()
        raise

def step3_upload_raw_dataset_from_rows(rows: list[dict[str, Any]]) -> tuple[str, str, dict[str, str]]:
    dataset_name = f"insurance-raw-{uuid.uuid4().hex[:8]}"
    out_rows: list[dict[str, Any]] = []
    id_map: dict[str, str] = {}
    with RAW_DATASET_PATH.open("w") as handle:
        for entry in rows:
            instr = (entry.get("instruction") or "").strip()
            if not instr:
                continue
            cid = uuid.uuid4().hex
            record = {
                "custom_id": cid,
                "prompt": instr,
                "messages": teacher_messages(instr),
                "base_completion": (entry.get("response") or "")[:4000],
                "category": entry.get("category", ""),
                "intent": entry.get("intent", ""),
            }
            handle.write(json.dumps(record) + "\n")
            out_rows.append(record)
            id_map[cid] = instr
    schema = [
        {"name": "custom_id", "type": {"name": "string"}},
        {"name": "prompt", "type": {"name": "string"}},
        {"name": "messages", "type": {"name": "json"}},
        {"name": "base_completion", "type": {"name": "string"}},
        {"name": "category", "type": {"name": "string"}},
        {"name": "intent", "type": {"name": "string"}},
    ]
    dataset = post_dataset_payload(dataset_name, schema, out_rows)
    print("Raw dataset ID:", dataset["id"])
    print("Raw dataset version:", dataset["current_version"])
    return dataset["id"], dataset["current_version"], id_map

def step4_run_batch_inference(source_dataset_id: str, source_dataset_version: str) -> tuple[str, str]:
    payload = {
        "type": "batch_inference",
        "src": [
            {
                "id": source_dataset_id,
                "version": source_dataset_version,
                "mapping": {
                    "type": "text_messages",
                    "messages": {"type": "column", "name": "messages"},
                    "custom_id": {"type": "column", "name": "custom_id"},
                    "max_tokens": {"type": "text", "value": "1024"},
                },
            }
        ],
        "dst": [],
        "params": {"model": TEACHER_MODEL, "completion_window": BATCH_COMPLETION_WINDOW},
    }
    operation = datalab_request("POST", "/v1/operations", json_body=payload).json()
    if not operation.get("dst"):
        raise RuntimeError(f"No destination dataset returned: {json.dumps(operation, indent=2)}")
    op_id = operation["id"]
    dst_dataset_id = operation["dst"][0]["id"]
    print("Operation:", op_id, "status=", operation.get("status"))
    print("Output dataset ID:", dst_dataset_id)
    while True:
        time.sleep(POLL_SECONDS)
        operation = datalab_request("GET", f"/v1/operations/{op_id}").json()
        status = operation["status"]
        print("Batch status:", status)
        if status in {"succeeded", "failed", "cancelled", "unknown"}:
            break
    if status != "succeeded":
        raise RuntimeError(f"Batch inference ended with status={status}")
    return op_id, dst_dataset_id

def step5_download_and_curate(output_dataset_id: str, id_map: dict[str, str]) -> Path:
    export_response = datalab_request(
        "GET",
        f"/v1/datasets/{output_dataset_id}/export",
        params={"format": "jsonl"},
    )
    RAW_BATCH_OUTPUT_PATH.write_text(export_response.text)
    records = [json.loads(line) for line in export_response.text.strip().splitlines() if line.strip()]
    written = 0
    with CURATED_TRAINING_PATH.open("w") as handle:
        for rec in records:
            prompt = extract_prompt_from_record(rec, id_map)
            reply = extract_reply_from_record(rec)
            if not prompt or not reply or len(reply) < 50:
                continue
            sample = {
                "messages": [
                    {"role": "system", "content": CLAIMS_SYSTEM_PROMPT},
                    {"role": "user", "content": prompt},
                    {"role": "assistant", "content": reply},
                ]
            }
            handle.write(json.dumps(sample) + "\n")
            written += 1
    if not records:
        raise RuntimeError(f"Batch output dataset {output_dataset_id} exported no rows.")
    if written == 0:
        raise RuntimeError(
            "No usable assistant replies. Inspect " + str(RAW_BATCH_OUTPUT_PATH)
        )
    print("Curated examples written:", written, "->", CURATED_TRAINING_PATH)
    return CURATED_TRAINING_PATH

def step6_upload_training_file(curated_path: Path) -> str:
    with curated_path.open("rb") as handle:
        file_obj = client.files.create(file=handle, purpose="fine-tune")
    print("Training file ID:", file_obj.id)
    return file_obj.id

def step7_launch_finetuning(train_file_id: str) -> str:
    job = client.fine_tuning.jobs.create(
        model=FINE_TUNE_BASE_MODEL,
        training_file=train_file_id,
        hyperparameters={
            "learning_rate": 2e-5,
            "n_epochs": N_EPOCHS,
            "lora": True,
            "lora_r": LORA_RANK,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
            "packing": True,
        },
    )
    print("Fine-tune job:", job.id, "status=", job.status)
    while True:
        time.sleep(POLL_SECONDS)
        job = client.fine_tuning.jobs.retrieve(job.id)
        print("Fine-tune status:", job.status, "trained_tokens:", getattr(job, "trained_tokens", 0))
        if job.status in {"succeeded", "failed", "cancelled"}:
            break
    if job.status != "succeeded":
        err = getattr(job, "error", "")
        raise RuntimeError(f"Fine-tuning failed: {err}")
    return job.id

def step8_deploy_lora(ft_job_id: str, adapter_name: str) -> dict[str, Any]:
    checkpoints = client.fine_tuning.jobs.checkpoints.list(ft_job_id).data
    if not checkpoints:
        raise RuntimeError("No checkpoints found for this job.")
    ckpt_id = checkpoints[-1].id
    attempt_errors: list[str] = []
    model_info: dict[str, Any] | None = None
    for base_model in deployment_base_models(ft_job_id):
        payload = {
            "source": f"{ft_job_id}:{ckpt_id}",
            "base_model": base_model,
            "name": adapter_name,
            "description": "Insurance claims chatbot — Nebius Data Lab + LoRA demo",
        }
        response = requests.post(f"{CONTROL_URL}/v0/models", json=payload, headers=HEADERS, timeout=120)
        if response.ok:
            model_info = response.json()
            print("Deploy accepted with base_model=", base_model)
            break
        body_text = response.text.strip() or "<empty body>"
        attempt_errors.append(f"base_model={base_model} status={response.status_code} body={body_text}")
        print("Deploy rejected for base_model=", base_model)
    if model_info is None:
        raise RuntimeError(
            "LoRA deployment failed for all candidate base models:\n- " + "\n- ".join(attempt_errors)
        )
    model_name = model_info["name"]
    print("Deploy initiated:", model_name)
    for _ in range(60):
        time.sleep(10)
        encoded_name = quote(model_name, safe="")
        response = requests.get(f"{CONTROL_URL}/v0/models/{encoded_name}", headers=HEADERS, timeout=60)
        response.raise_for_status()
        info = response.json()
        status = info.get("status", "?")
        print("Deploy status:", status)
        if status == "active":
            return info
        if status == "error":
            raise RuntimeError(f"Deployment failed: {info.get('status_reason')}")
    raise TimeoutError("Model did not become active in time.")

print("Helper functions loaded.")


## Step 1 — Load dataset and build train / test split

**What we are doing:** Load [Bitext Insurance LLM Chatbot](https://huggingface.co/datasets/bitext/Bitext-insurance-llm-chatbot-training-dataset), hold out **10** diverse prompts, and sample **training** rows (default **~32**, stratified by `category`).

**Why we do it:** You need a fixed **test set** to compare base vs fine-tuned models fairly, and a **training** sample for Data Lab + teacher + LoRA.

**In simple words:** Pick homework questions (test) and study notes (train) from the same book.


In [ ]:
random.seed(RANDOM_SEED)

print("Loading dataset from Hugging Face …")
ds = load_dataset("bitext/Bitext-insurance-llm-chatbot-training-dataset", split="train")
rows = [ds[i] for i in range(len(ds))]
print("Total rows:", len(rows))

# Hold out 10 prompts — one random row per category until we have 10 categories
by_cat: dict[str, list[dict[str, Any]]] = defaultdict(list)
for r in rows:
    by_cat[str(r.get("category", "UNKNOWN"))].append(dict(r))

rng = random.Random(RANDOM_SEED)
categories = sorted(by_cat.keys())
rng.shuffle(categories)

test_rows: list[dict[str, Any]] = []
used_instr = set()
for cat in categories:
    if len(test_rows) >= 10:
        break
    pool = [x for x in by_cat[cat] if (x.get("instruction") or "").strip()]
    if not pool:
        continue
    pick = rng.choice(pool)
    instr = (pick.get("instruction") or "").strip()
    if instr in used_instr:
        continue
    used_instr.add(instr)
    test_rows.append(pick)

# Fallback if fewer than 10 categories with data
while len(test_rows) < 10:
    pick = rng.choice(rows)
    d = dict(pick) if not isinstance(pick, dict) else pick
    instr = (d.get("instruction") or "").strip()
    if instr and instr not in used_instr:
        used_instr.add(instr)
        test_rows.append(d)

test_instructions = [(r.get("instruction") or "").strip() for r in test_rows]

# Remove test rows from pool (match on instruction text)
test_set = set(test_instructions)
pool = [dict(r) for r in rows if (r.get("instruction") or "").strip() not in test_set]
print("Pool after holdout:", len(pool))

# Stratified sample SAMPLE_SIZE from pool
buckets: dict[str, list[dict[str, Any]]] = defaultdict(list)
for r in pool:
    buckets[str(r.get("category", "UNKNOWN"))].append(r)

for b in buckets.values():
    rng.shuffle(b)

train_sample: list[dict[str, Any]] = []
cat_order = list(buckets.keys())
rng.shuffle(cat_order)
idx = 0
while len(train_sample) < SAMPLE_SIZE and any(buckets.values()):
    for c in cat_order:
        if len(train_sample) >= SAMPLE_SIZE:
            break
        if buckets[c]:
            train_sample.append(buckets[c].pop())
    idx += 1
    if idx > SAMPLE_SIZE * 3:
        break

print("Training sample size:", len(train_sample))
print("Test prompts:", len(test_instructions))
print("Category counts (train):", dict(Counter(r.get("category", "?") for r in train_sample).most_common(8)))

TEST_PROMPTS_PATH.write_text(json.dumps(test_instructions, indent=2))
TRAINING_SAMPLE_META_PATH.write_text(json.dumps({"n_train": len(train_sample), "n_test": len(test_instructions)}, indent=2))
TRAIN_SAMPLE_PATH.write_text(json.dumps(train_sample, ensure_ascii=False))

state = load_state()
state.update({"last_completed_step": 1, "n_train": len(train_sample), "n_test": len(test_instructions)})
save_state(state)

test_instructions[:3]

## Step 2 — BEFORE: base model on held-out questions

**What we are doing:** Call the base **chat** model (`CHAT_BASE_MODEL`, Meta-prefixed id) on each of the 10 test prompts with the same **system prompt** you will use after fine-tuning.

**Why we do it:** This is your **baseline**. Step 8 and Step 9 compare **before** vs **after** on identical inputs.

**In simple words:** Save “what the out-of-the-box model says” before you teach it anything new.


In [ ]:
if not TEST_PROMPTS_PATH.exists():
    raise RuntimeError("Run Step 1 first.")

test_instructions = json.loads(TEST_PROMPTS_PATH.read_text())

before_answers: list[dict[str, str]] = []
for prompt in test_instructions:
    resp = client.chat.completions.create(
        model=CHAT_BASE_MODEL,
        messages=[
            {"role": "system", "content": CLAIMS_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        max_tokens=512,
    )
    ans = (resp.choices[0].message.content or "").strip()
    before_answers.append({"prompt": prompt, "answer": ans})
    print("✓", prompt[:80])

BEFORE_ANSWERS_PATH.write_text(json.dumps(before_answers, indent=2))
state = load_state()
state.update({"last_completed_step": 2})
save_state(state)
print("Saved:", BEFORE_ANSWERS_PATH)
before_answers[:1]

## Step 3 — Upload training sample to Data Lab

**What we are doing:** Build JSONL rows with `messages` for the **teacher** (system + user), upload them as a Data Lab dataset (`POST /v1/datasets`), and persist `insurance_raw_dataset.jsonl` for reproducible joins.

**Why we do it:** Batch inference needs a **versioned dataset** and a stable mapping from each row’s `custom_id` back to the original user prompt—used when curating teacher outputs.

**In simple words:** Package your training prompts so the 70B model can answer them offline, in bulk.


In [ ]:
if TRAIN_SAMPLE_PATH.exists():
    train_sample = json.loads(TRAIN_SAMPLE_PATH.read_text())
elif "train_sample" in globals() and train_sample:
    pass
else:
    raise RuntimeError("Run Step 1 first (train_sample.json missing).")

if REUSE_OUTPUT_DATASET_ID:
    state = load_state()
    raw_dataset_id = state["raw_dataset_id"]
    raw_dataset_version = state["raw_dataset_version"]
    id_map = load_id_map_from_artifact()
    print("Skipping Step 3 upload (REUSE_OUTPUT_DATASET_ID): using synced raw JSONL + id_map.")
else:
    raw_dataset_id, raw_dataset_version, id_map = step3_upload_raw_dataset_from_rows(train_sample)
state = load_state()
state.update(
    {
        "raw_dataset_id": raw_dataset_id,
        "raw_dataset_version": raw_dataset_version,
        "last_completed_step": 3,
    }
)
save_state(state)
{"raw_dataset_id": raw_dataset_id, "raw_dataset_version": raw_dataset_version}

## Step 4 — Teacher batch inference (70B)

Runs **Data Lab batch inference** on the uploaded dataset. This step can take a long time.

**Reuse a previous successful batch (no new Data Lab job):** keep `insurance_raw_dataset.jsonl` from that run (same rows / `id_map` as the batch). Then either set `REUSE_OUTPUT_DATASET_ID` in `.env` to the teacher output dataset id, or set `REUSE_BATCH_FROM_STATE=1` to reuse `output_dataset_id` from `insurance_notebook_state.json`. Do **not** re-run Step 3 with new data if you reuse an old batch output.

In [ ]:
if "raw_dataset_id" not in globals() or "raw_dataset_version" not in globals():
    state = load_state()
    raw_dataset_id = state["raw_dataset_id"]
    raw_dataset_version = state["raw_dataset_version"]

reuse_id = REUSE_OUTPUT_DATASET_ID
if not reuse_id and REUSE_BATCH_FROM_STATE:
    reuse_id = (load_state().get("output_dataset_id") or "").strip()

if reuse_id:
    if not RAW_DATASET_PATH.is_file():
        raise RuntimeError(
            "Reusing batch output requires insurance_raw_dataset.jsonl from the same run. "
            "Restore insurance_claims_demo_artifacts/ or re-run Steps 1–3 before curating."
        )
    output_dataset_id = reuse_id
    operation_id = load_state().get("batch_operation_id", "reused")
    print("Skipping new Data Lab batch inference; using existing teacher output dataset:", output_dataset_id)
else:
    operation_id, output_dataset_id = step4_run_batch_inference(raw_dataset_id, raw_dataset_version)

state = load_state()
state.update(
    {
        "batch_operation_id": operation_id,
        "output_dataset_id": output_dataset_id,
        "last_completed_step": 4,
    }
)
save_state(state)
print("Teacher output dataset:", output_dataset_id)

## Step 5 — Download and curate training JSONL

**What we are doing:** Export the teacher output dataset (`GET .../export?format=jsonl`), parse each record, pair **prompt** with **assistant text**, and write `insurance_curated_training.jsonl` in **chat** format (`system` / `user` / `assistant`).

**Why we do it:** The fine-tuning API expects **conversation** JSONL; raw batch rows can include extra fields and need normalization.

**In simple words:** Turn the teacher’s answers into homework answer keys for the student model.


In [ ]:
if "output_dataset_id" not in globals():
    output_dataset_id = load_state()["output_dataset_id"]
if "id_map" not in globals():
    id_map = load_id_map_from_artifact()

curated_path = step5_download_and_curate(output_dataset_id, id_map)
state = load_state()
state.update({"curated_path": str(curated_path), "last_completed_step": 5})
save_state(state)
curated_path

## Step 6 — Upload training file and launch LoRA fine-tuning

**What we are doing:** Upload the curated JSONL (`purpose="fine-tune"`), then create a **LoRA** job with rank/alpha/dropout and `n_epochs`.

**Why we do it:** LoRA updates a small adapter instead of full weights—faster and cheaper for demos; you still get a measurable style/behavior shift on insurance prompts.

**In simple words:** Train a lightweight “add-on” brain on top of the 8B base model.


In [ ]:
if "curated_path" not in globals():
    curated_path = Path(load_state()["curated_path"])

training_file_id = step6_upload_training_file(curated_path)
ft_job_id = step7_launch_finetuning(training_file_id)
state = load_state()
state.update({"training_file_id": training_file_id, "ft_job_id": ft_job_id, "last_completed_step": 6})
save_state(state)
ft_job_id

## Step 7 — Deploy custom LoRA endpoint

**What we are doing:** Pick the latest checkpoint, call **`POST /v0/models`** with `source`, `base_model`, and adapter `name`, then poll until status is **active**.

**Why we do it:** This exposes your adapter as a **serverless** private model name you can pass to `chat.completions`—same API as the base model, different `model` string.

**In simple words:** Publish your trained adapter so apps can call it like any other model id.

**Billing note:** Serverless adapters are typically billed **per token** when invoked; idle time does not reserve a dedicated GPU.


In [ ]:
if "ft_job_id" not in globals():
    ft_job_id = load_state()["ft_job_id"]

deploy_info = step8_deploy_lora(ft_job_id, ADAPTER_NAME)
deployed_model_name = deploy_info["name"]
print()
print("=" * 60)
print("CUSTOM_MODEL_NAME for app.py / .env:")
print(deployed_model_name)
print("=" * 60)

state = load_state()
state.update({"deployed_model_name": deployed_model_name, "adapter_name": ADAPTER_NAME, "last_completed_step": 7})
save_state(state)
deploy_info

## Step 8 — AFTER: fine-tuned model on the same 10 prompts

**What we are doing:** Re-run the 10 held-out prompts using **`deployed_model_name`** from Step 7 (your LoRA) with the same system prompt as Step 2.

**Why we do it:** Apples-to-apples comparison: same inputs and prompt framing, different weights.

**In simple words:** Ask the same questions again and see whether answers improved.


In [ ]:
if "deployed_model_name" not in globals():
    deployed_model_name = load_state()["deployed_model_name"]
if not TEST_PROMPTS_PATH.exists():
    raise RuntimeError("Missing test prompts. Run Step 1.")

test_instructions = json.loads(TEST_PROMPTS_PATH.read_text())
after_answers: list[dict[str, str]] = []
for prompt in test_instructions:
    resp = client.chat.completions.create(
        model=deployed_model_name,
        messages=[
            {"role": "system", "content": CLAIMS_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        max_tokens=512,
    )
    ans = (resp.choices[0].message.content or "").strip()
    after_answers.append({"prompt": prompt, "answer": ans})
    print("✓", prompt[:80])

AFTER_ANSWERS_PATH.write_text(json.dumps(after_answers, indent=2))
state = load_state()
state.update({"last_completed_step": 8})
save_state(state)
after_answers[:1]

## Step 9 — Before vs After vs dataset reference

**What we are doing:** Print side-by-side excerpts for each test prompt: **base 8B**, **fine-tuned**, and the **original dataset** response (truncated) when the instruction matches a training row.

**Why we do it:** Helps you sanity-check whether the model is drifting from reference text or learning a new style—useful when iterating on `SAMPLE_SIZE`, teacher, or epochs.

**In simple words:** Three-way diff: generic model, your adapter, and the source material.


In [ ]:
before_list = json.loads(BEFORE_ANSWERS_PATH.read_text())
after_list = json.loads(AFTER_ANSWERS_PATH.read_text())
# Ground truth from original test rows
if not TEST_PROMPTS_PATH.exists():
    raise RuntimeError("Missing test prompts")

test_instructions = json.loads(TEST_PROMPTS_PATH.read_text())
ds = load_dataset("bitext/Bitext-insurance-llm-chatbot-training-dataset", split="train")
lookup = {}
for i in range(len(ds)):
    row = ds[i]
    ins = (row.get("instruction") or "").strip()
    if ins:
        lookup[ins] = (row.get("response") or "")[:1200]

for i, prompt in enumerate(test_instructions):
    b = next((x["answer"] for x in before_list if x["prompt"] == prompt), "")
    a = next((x["answer"] for x in after_list if x["prompt"] == prompt), "")
    gt = lookup.get(prompt, "(not found in dataset)")
    print("\n" + "=" * 70)
    print(f"Q{i+1}:", prompt[:200], "…" if len(prompt) > 200 else "")
    print("\n--- BEFORE (base 8B) ---\n", b[:900], "…" if len(b) > 900 else "")
    print("\n--- AFTER (fine-tuned) ---\n", a[:900], "…" if len(a) > 900 else "")
    print("\n--- Original dataset response (truncated) ---\n", gt[:500], "…" if len(gt) > 500 else "")

## Cleanup — delete deployed custom model (optional)

Serverless LoRA is billed per token; deleting removes the private endpoint. There is no **disable** API — only **delete**.

In [ ]:
# Uncomment to delete the deployed model when you are completely done (including Gradio demo).

# if "deployed_model_name" not in globals():
#     deployed_model_name = load_state().get("deployed_model_name")
# if deployed_model_name:
#     enc = quote(deployed_model_name, safe="")
#     r = requests.delete(f"{CONTROL_URL}/v0/models/{enc}", headers=HEADERS, timeout=60)
#     print(r.status_code, r.text[:500])
# else:
#     print("No deployed_model_name in state.")

## Next steps

1. Copy **`CUSTOM_MODEL_NAME`** from Step 7 into `.env` next to `app.py` (see `.env.example`).
2. Run `python app.py` for the Gradio chat (**Base**, **Fine-tuned**, **Side-by-side**).
3. Read [README.md](README.md) for setup, diagrams, and references.

Docs: [Token Factory](https://docs.tokenfactory.nebius.com/), [Deploy custom LoRA](https://docs.tokenfactory.nebius.com/fine-tuning/deploy-custom-model), [Data Lab](https://docs.tokenfactory.nebius.com/data-lab/overview).
